# Notebook 42 - Does unclipped CIC-IoMT-2024 carry the same extreme rows?

**Why this notebook.** Notebook 35 isolated the BatchNorm collapse on CICIoT2023 to single training rows whose standardised feature value reaches 1,390 (about one row in 17,000 has any value above 100). Notebook 34 found no collapse on CIC-IoMT-2024 under the same recipe, and the one preprocessing difference that bears on the mechanism is that the IoMT pipeline clips every feature to its 0.1 and 99.9 training percentiles. Test T1 then showed that clipping removes every CICIoT2023 collapse. What is still missing is the other half of the control: whether the IoMT corpus, *without* clipping, contains rows as extreme as CICIoT2023's. If it does, clipping is what protects it and the paper's claim stands as written. If it does not, the corpus is benign with or without clipping and the claim must be softened to "consistent with".

**What it does.** CPU only, no model. The IoMT training rows are loaded, deduplicated and split exactly as `src/saber/bridge_iomt.py` does (same loader, same dedup rule, same seeded carve-out; the official test files are never opened), then standardised with their own train-fitted mean and standard deviation and **no clipping**. The notebook reports the per-feature maximum absolute standardised value, the fraction of rows with any value above 10 and above 100, the largest value and its feature, and the one-row batch-variance bound $1 + z^2/1024$ of equation (5) in the paper. The same quantities are computed analytically for the clipped pipeline from the committed bridge manifest (clip bounds and scaler), and the CICIoT2023 numbers of notebook 35 are placed beside them.

**Reading, fixed in stage 2 before any data is opened.** U1 holds if at least one unclipped IoMT training row has an absolute standardised value above 100; the verdict is written either way.

In [ ]:
# Stage 1 - bootstrap
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, gc, time, subprocess, importlib
from pathlib import Path
import numpy as np, pandas as pd

REPO = Path("/content/drive/MyDrive/SABER_Research/saber-ids")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
importlib.invalidate_caches()
import src.saber.bridge_iomt as BI

R = REPO / "results/saber"
OUT = R / "42_iomt_extreme_values"; OUT.mkdir(parents=True, exist_ok=True)
DATA_CACHE = REPO / "data/iomt_bridge"                      # written by notebook 32 (git-ignored tensors, committed manifest copy below)
CICIOT_DIAG = R / "35_trigger_isolation/extreme_value_diagnostic.json"
print("repo:", REPO, "| IoMT loader dir:", BI.IOMT_SRC)
print("cache manifest present:", (DATA_CACHE / "manifest.json").exists(), "| CICIoT2023 diagnostic present:", CICIOT_DIAG.exists())

In [ ]:
# Stage 2 - pre-registration (frozen: rerunning this cell can never change it after its commit)
PREREG = {
    "notebook": "42_iomt_extreme_values",
    "question": ("does the CIC-IoMT-2024 training corpus, standardised without clipping, contain rows as extreme as the "
                 "CICIoT2023 rows that trigger the BatchNorm collapse (absolute standardised value above 100)"),
    "motivation": ("notebook 34 found no normalisation collapse on CIC-IoMT-2024 under the CICIoT2023 recipe; the IoMT pipeline "
                   "clips every feature to its 0.1/99.9 training percentiles, and notebook 35 (T1) showed clipping removes every "
                   "CICIoT2023 collapse; the missing control is whether unclipped IoMT has the trigger rows at all"),
    "design": {
        "corpus": ("CIC-IoMT-2024 WiFi/MQTT, official training files only, exact duplicates removed, seeded 20% validation carve-out, "
                   "exactly as src/saber/bridge_iomt.build_bridge; the official test files are never opened; training rows only"),
        "unclipped": "features standardised with the train-fitted mean and standard deviation of the UNCLIPPED training rows; no clipping",
        "clipped": "the committed bridge manifest (clip_low, clip_high, scaler_mean, scaler_std) gives the clipped pipeline's bounds analytically",
        "quantities": ["per-feature maximum |z|", "number of features with maximum |z| > 100", "fraction of rows with any |z| > 10 and > 100",
                        "largest |z| and its feature", "one-row batch-variance bound 1 + z^2/1024 for the largest value",
                        "the same for the clipped pipeline", "CICIoT2023 reference values from notebook 35"],
        "reproduction_check": "the number of training rows must equal manifest n_train (3,612,063) or the notebook stops",
    },
    "claims": {
        "U1": "at least one unclipped IoMT training row has an absolute standardised value above 100",
    },
    "readings": {
        "U1_held": "unclipped_has_extreme_rows: clipping is what protects the IoMT runs; the paper's generality sentence stands",
        "U1_not_held": "unclipped_has_no_extreme_rows: the IoMT corpus is benign with or without clipping; soften the sentence to 'consistent with'",
    },
    "reporting_rule": "U1 reported as held or not held; every quantity written to the result files either way",
    "no_test_access": True, "no_new_selection": True, "no_model": True, "cpu_only": True,
}
pre_path = OUT / "U42_PREREGISTRATION.json"
if pre_path.exists():
    assert json.loads(pre_path.read_text()) == PREREG, "pre-registration on disk differs from this cell; do not edit a committed pre-registration"
    print("pre-registration already on disk and identical")
else:
    pre_path.write_text(json.dumps(PREREG, indent=2)); print("pre-registration written:", pre_path)

In [ ]:
# Stage 2b - commit the pre-registration before any result exists (skipped automatically once results exist)
def _prereg_committed():
    rel = "results/saber/42_iomt_extreme_values/U42_PREREGISTRATION.json"
    h = subprocess.run(["git", "log", "-1", "--format=%h", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    dirty = subprocess.run(["git", "status", "--porcelain", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    return bool(h) and not dirty


_results_exist = any(p.is_file() and p.name != "U42_PREREGISTRATION.json" for p in OUT.rglob("*"))
if _results_exist:
    assert _prereg_committed(), ("results exist but the pre-registration is not committed, so it can no longer be committed before any "
                                 "result; stop here and report it")
    print("results already exist in", OUT.name, "and the pre-registration is committed; nothing to do here")
else:
    os.chdir(REPO)
    r = subprocess.run(["python", "tools/commit_cell.py",
                        "NB42 pre-registration, committed before any result: does unclipped CIC-IoMT-2024 carry rows with absolute "
                        "standardised value above 100, as CICIoT2023 does (max 1,390)? Training rows loaded, deduplicated and split exactly "
                        "as the IoMT bridge, standardised without clipping; per-feature max |z|, fraction of rows above 10 and 100, "
                        "one-row batch-variance bound; clipped pipeline computed analytically from the bridge manifest; CPU only, no model, "
                        "no test access. U1 = at least one unclipped row above 100; readings unclipped_has_extreme_rows / "
                        "unclipped_has_no_extreme_rows."], capture_output=True, text=True)
    print(r.stdout); print(r.stderr)

In [ ]:
# Stage 3 - the diagnostic, and a self-test on synthetic data before any real row is opened
def extreme_value_diagnostic(X, thresholds=(10.0, 100.0), chunk=250_000):
    # X: float array (n, d) of UNCLIPPED features. Standardises with X's own mean/std (std < 1e-8 -> 1) and returns the
    # per-feature max |z|, counts of rows with any |z| above each threshold, and the global maximum with its feature and row.
    X = np.asarray(X)
    n, d = X.shape
    mean = X.mean(axis=0, dtype=np.float64)
    std = X.std(axis=0, dtype=np.float64)
    std = np.where(std < 1e-8, 1.0, std)
    max_abs_z = np.zeros(d, dtype=np.float64)
    rows_above = {float(t): 0 for t in thresholds}
    gmax, gfeat, grow = 0.0, -1, -1
    for s in range(0, n, chunk):
        Z = np.abs((X[s:s + chunk].astype(np.float64) - mean) / std)
        max_abs_z = np.maximum(max_abs_z, Z.max(axis=0))
        rowmax = Z.max(axis=1)
        for t in thresholds:
            rows_above[float(t)] += int((rowmax > t).sum())
        j = int(rowmax.argmax())
        if rowmax[j] > gmax:
            gmax, grow, gfeat = float(rowmax[j]), s + j, int(Z[j].argmax())
    return {"n_rows": int(n), "n_features": int(d), "mean": mean.tolist(), "std": std.tolist(),
            "max_abs_z_per_feature": max_abs_z.tolist(),
            "features_with_max_abs_z_over_100": int((max_abs_z > 100).sum()),
            "features_with_max_abs_z_over_10": int((max_abs_z > 10).sum()),
            "rows_any_abs_z_over_10": rows_above[10.0], "rows_any_abs_z_over_100": rows_above[100.0],
            "fraction_rows_any_abs_z_over_10": rows_above[10.0] / n, "fraction_rows_any_abs_z_over_100": rows_above[100.0] / n,
            "max_abs_z": gmax, "max_abs_z_feature_index": gfeat, "max_abs_z_row_index": int(grow),
            "one_row_batch_variance_bound_1024": 1.0 + gmax ** 2 / 1024.0}


def clipped_pipeline_bounds(manifest):
    # Largest |z| the clipped pipeline can produce, per feature, from its committed clip bounds and scaler.
    lo, hi = np.asarray(manifest["clip_low"], float), np.asarray(manifest["clip_high"], float)
    mean, std = np.asarray(manifest["scaler_mean"], float), np.asarray(manifest["scaler_std"], float)
    z = np.maximum(np.abs((lo - mean) / std), np.abs((hi - mean) / std))
    return {"max_abs_z_per_feature": z.tolist(), "max_abs_z": float(z.max()), "features_with_max_abs_z_over_100": int((z > 100).sum()),
            "features_with_max_abs_z_over_10": int((z > 10).sum()), "one_row_batch_variance_bound_1024": 1.0 + float(z.max()) ** 2 / 1024.0}


# self-test: a unit-variance block plus one planted row; the planted row must be found with the right feature and |z|
rng = np.random.default_rng(0)
Xs = rng.normal(size=(50_000, 6)).astype(np.float32)
Xs[12_345, 4] = 400.0                                   # one extreme value in feature 4
Xs[:, 5] = 3.0                                          # a constant feature (std -> 1)
diag = extreme_value_diagnostic(Xs, chunk=7_000)
assert diag["max_abs_z_feature_index"] == 4 and diag["max_abs_z_row_index"] == 12_345, diag
assert diag["rows_any_abs_z_over_100"] == 1 and diag["features_with_max_abs_z_over_100"] == 1
assert 150 < diag["max_abs_z"] < 400 and abs(diag["max_abs_z_per_feature"][5]) < 1e-6
m_fake = {"clip_low": [-1, -1], "clip_high": [1, 9], "scaler_mean": [0, 0], "scaler_std": [0.5, 1]}
assert clipped_pipeline_bounds(m_fake)["max_abs_z"] == 9.0
print("self-test passed: planted row found at", diag["max_abs_z_row_index"], "feature", diag["max_abs_z_feature_index"],
      "| |z| =", round(diag["max_abs_z"], 1), "| chunking consistent")

In [ ]:
# Stage 4 - bridge manifest (the cached full copy if present, else the committed copy) and the CICIoT2023 reference
cached = DATA_CACHE / "manifest.json"
MANIFEST = json.loads(cached.read_text()) if cached.exists() else json.loads((R / "32_iomt_bridge/bridge_manifest.json").read_text())
assert MANIFEST["clip_percentiles"] == [0.1, 99.9], MANIFEST["clip_percentiles"]
HAS_BOUNDS = all(k in MANIFEST for k in ("clip_low", "clip_high", "scaler_mean", "scaler_std"))
cic = json.loads(CICIOT_DIAG.read_text())
print("manifest source:", "cached (full)" if cached.exists() else "committed copy", "| clip bounds in manifest:", HAS_BOUNDS,
      "| n_train", MANIFEST["n_train"], "| features", MANIFEST["n_features"])
print("CICIoT2023 (NB35, unclipped): max |z| =", round(max(cic["max_abs_z_per_feature"]), 1), "| features over 100:", cic["features_with_max_abs_z_over_100"],
      "| fraction of rows over 100:", cic["fraction_rows_any_abs_z_over_100"], "| over 10:", round(cic["fraction_rows_any_abs_z_over_10"], 4))

In [ ]:
# Stage 5 - unclipped IoMT training rows: load, deduplicate and split exactly as the bridge, then standardise WITHOUT clipping
t0 = time.time()
if BI.IOMT_SRC not in sys.path:
    sys.path.insert(0, BI.IOMT_SRC)
import iomtc_data as D                                   # the IoMT project's loader, used as-is (same as the bridge)

train_df, _test_unused = D.load_official(add_targets=True)
del _test_unused; gc.collect()                           # the official test files are never touched
feat_cols = [c for c in D.feature_columns(train_df)]
drop, leak_note = BI._leakage_drop_list()
feat_cols = [c for c in feat_cols if c not in set(drop)]
assert feat_cols == MANIFEST["feature_columns"], "feature columns differ from the committed bridge manifest; stop"
n_raw = int(len(train_df))
train_df = train_df.drop_duplicates(subset=feat_cols + ["attack_type"]).reset_index(drop=True)
n_dedup = int(len(train_df))
assert n_raw == MANIFEST["n_official_train_rows"] and n_dedup == MANIFEST["n_after_exact_dedup"], (n_raw, n_dedup)

rng = np.random.default_rng(BI.SEED)
val_mask = np.zeros(n_dedup, dtype=bool)
for _, idx in train_df.groupby("attack_type").indices.items():
    idx = np.asarray(idx); rng.shuffle(idx)
    val_mask[idx[: int(round(BI.VAL_FRACTION * len(idx)))]] = True
X_tr_raw = train_df.loc[~val_mask, feat_cols].to_numpy(np.float32)
y_tr = train_df.loc[~val_mask, "attack_type"].astype(str).to_numpy()
del train_df; gc.collect()
assert X_tr_raw.shape[0] == MANIFEST["n_train"], (X_tr_raw.shape, MANIFEST["n_train"])   # reproduction check of the pre-registration
print("training rows reproduced:", X_tr_raw.shape, "| load+split", round(time.time() - t0, 1), "s")

unclipped = extreme_value_diagnostic(X_tr_raw)
unclipped["max_abs_z_feature"] = feat_cols[unclipped["max_abs_z_feature_index"]]
unclipped["max_abs_z_row_class"] = str(y_tr[unclipped["max_abs_z_row_index"]])

# the clipped pipeline, computed from the same rows: clip at the 0.1/99.9 training percentiles, standardise, bound the largest |z|
lo = np.percentile(X_tr_raw, 0.1, axis=0).astype(np.float64); hi = np.percentile(X_tr_raw, 99.9, axis=0).astype(np.float64)
mean_c = np.zeros(len(feat_cols)); sq_c = np.zeros(len(feat_cols)); n_c = 0
for s_ in range(0, X_tr_raw.shape[0], 250_000):
    Xc = np.clip(X_tr_raw[s_:s_ + 250_000].astype(np.float64), lo, hi)
    mean_c += Xc.sum(axis=0); sq_c += (Xc ** 2).sum(axis=0); n_c += Xc.shape[0]
mean_c /= n_c; std_c = np.sqrt(np.maximum(sq_c / n_c - mean_c ** 2, 0.0)); std_c = np.where(std_c < 1e-8, 1.0, std_c)
clipped = clipped_pipeline_bounds({"clip_low": lo, "clip_high": hi, "scaler_mean": mean_c, "scaler_std": std_c})
if HAS_BOUNDS:                                           # cross-check against the bridge's own bounds when the cached manifest is present
    ref = clipped_pipeline_bounds(MANIFEST)
    print("cross-check vs bridge manifest: max |z| here", round(clipped["max_abs_z"], 3), "| bridge", round(ref["max_abs_z"], 3))
# which classes hold the rows above 100 (descriptive)
Z_rowmax = np.zeros(X_tr_raw.shape[0], dtype=np.float32)
mean = np.asarray(unclipped["mean"]); std = np.asarray(unclipped["std"])
for s in range(0, X_tr_raw.shape[0], 250_000):
    Z_rowmax[s:s + 250_000] = np.abs((X_tr_raw[s:s + 250_000].astype(np.float64) - mean) / std).max(axis=1)
over100 = pd.Series(y_tr[Z_rowmax > 100]).value_counts()
unclipped["rows_over_100_by_class"] = {str(k): int(v) for k, v in over100.items()}
del X_tr_raw; gc.collect()
print("unclipped IoMT: max |z| =", round(unclipped["max_abs_z"], 1), "in feature", unclipped["max_abs_z_feature"], "(class", unclipped["max_abs_z_row_class"] + ")",
      "| features over 100:", unclipped["features_with_max_abs_z_over_100"], "| rows over 100:", unclipped["rows_any_abs_z_over_100"],
      "(fraction", f"{unclipped['fraction_rows_any_abs_z_over_100']:.2e})", "| rows over 10:", f"{unclipped['fraction_rows_any_abs_z_over_10']:.4f}",
      "| one-row variance bound:", round(unclipped["one_row_batch_variance_bound_1024"], 1))

In [ ]:
# Stage 6 - result files and verdict (U1 reported as held or not held)
per_feature = pd.DataFrame({"feature": feat_cols,
                            "max_abs_z_unclipped": unclipped["max_abs_z_per_feature"],
                            "max_abs_z_clipped_pipeline": clipped["max_abs_z_per_feature"]})
per_feature.to_csv(OUT / "per_feature_max_abs_z.csv", index=False)
U1 = unclipped["rows_any_abs_z_over_100"] >= 1
verdict = {
    "arm": "U_unclipped_iomt_extreme_values",
    "U1_unclipped_row_above_100": bool(U1),
    "reading": "unclipped_has_extreme_rows" if U1 else "unclipped_has_no_extreme_rows",
    "unclipped_iomt": {k: v for k, v in unclipped.items() if k not in ("mean", "std", "max_abs_z_per_feature")},
    "clipped_iomt_pipeline": {k: v for k, v in clipped.items() if k != "max_abs_z_per_feature"},
    "ciciot2023_reference_nb35": {"max_abs_z": float(max(cic["max_abs_z_per_feature"])), "features_with_max_abs_z_over_100": cic["features_with_max_abs_z_over_100"],
                                  "fraction_rows_any_abs_z_over_100": cic["fraction_rows_any_abs_z_over_100"], "fraction_rows_any_abs_z_over_10": cic["fraction_rows_any_abs_z_over_10"],
                                  "one_row_batch_variance_bound_1024": 1.0 + float(max(cic["max_abs_z_per_feature"])) ** 2 / 1024.0},
    "rows": {"official_train": n_raw, "after_dedup": n_dedup, "train_side": MANIFEST["n_train"]},
    "leakage_note": leak_note,
    "prereg": PREREG,
}
(OUT / "U42_verdict.json").write_text(json.dumps(verdict, indent=2))
print(json.dumps({k: verdict[k] for k in ("U1_unclipped_row_above_100", "reading")}, indent=1))
print("unclipped:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in verdict["unclipped_iomt"].items() if k != "rows_over_100_by_class"})
print("rows over 100 by class:", verdict["unclipped_iomt"]["rows_over_100_by_class"])

In [ ]:
# Stage 7 - commit the results (standard commit cell; the message is filled in from the verdict)
v = json.load(open(OUT / "U42_verdict.json"))
pre = subprocess.run(["git", "log", "-1", "--format=%h", "--", "results/saber/42_iomt_extreme_values/U42_PREREGISTRATION.json"], capture_output=True, text=True, cwd=REPO).stdout.strip() or "see log"
u, c, ref = v["unclipped_iomt"], v["clipped_iomt_pipeline"], v["ciciot2023_reference_nb35"]
msg = (f"NB42 results (pre-registration {pre}): U1 {'HELD' if v['U1_unclipped_row_above_100'] else 'NOT HELD'} -> {v['reading']}. "
       f"Unclipped CIC-IoMT-2024 training rows ({v['rows']['train_side']:,}): max |z| {u['max_abs_z']:.1f} in {u['max_abs_z_feature']} "
       f"(class {u['max_abs_z_row_class']}); {u['features_with_max_abs_z_over_100']} of {u['n_features']} features above 100; "
       f"rows with any |z| above 100: {u['rows_any_abs_z_over_100']} ({u['fraction_rows_any_abs_z_over_100']:.2e}), above 10: {u['fraction_rows_any_abs_z_over_10']:.4f}; "
       f"one-row batch-variance bound {u['one_row_batch_variance_bound_1024']:.0f}. Clipped IoMT pipeline (analytic from the manifest): "
       f"max |z| {c['max_abs_z']:.2f}, bound {c['one_row_batch_variance_bound_1024']:.2f}. CICIoT2023 reference (NB35): max |z| {ref['max_abs_z']:.0f}, "
       f"{ref['features_with_max_abs_z_over_100']} features above 100, fraction of rows above 100 {ref['fraction_rows_any_abs_z_over_100']:.2e}.")
print(msg)
os.chdir(REPO)
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout); print(r.stderr)